In [1]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_finance import candlestick_ohlc
import matplotlib.dates as mdates
from binance import Client

/root/miniconda3/envs/finance/lib/python3.9/site-packages/mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


In [2]:
client = Client()

def get_sample(coin, start_date="1 Jan, 2020"):
    klines = np.array(client.get_historical_klines("ETHUSDT", Client.KLINE_INTERVAL_12HOUR, start_date))
    sample = pd.DataFrame(klines.reshape(-1, 12), dtype=float, columns=['datetime', 
                                                                        'open', 
                                                                        'high', 
                                                                        'low', 
                                                                        'close', 
                                                                        'volume', 
                                                                        'close time', 
                                                                        'quote asset volume, number of trades', 
                                                                        'number of trades',
                                                                        'taker buy base asset volume', 
                                                                        'taker buy quote asset volume', 
                                                                        'ignore'])
    sample['datetime'] = pd.to_datetime(sample['datetime'], unit='ms')
    sample.set_index('datetime', inplace=True)
    sample = sample[['open', 'high', 'low', 'close', 'volume', 'number of trades']]
    return sample

sample = get_sample('BTCUSDT', start_date="1 Aug, 2021")
sample.tail()


,open,high,low,close,volume,number of trades
datetime,,,,,,
2021-10-23 00:00:00,3972.19,4052.89,3937.51,4039.20,147345.8332,408140.0
2021-10-23 12:00:00,4039.24,4172.00,4017.18,4167.12,179231.1269,532751.0
2021-10-24 00:00:00,4167.13,4186.57,4024.01,4026.20,140733.6533,389365.0
2021-10-24 12:00:00,4026.21,4099.98,3961.50,4082.33,209382.1853,528604.0
2021-10-25 00:00:00,4082.33,4154.93,4067.71,4131.91,75358.2675,221060.0


In [3]:
info = client.get_all_tickers()
coins = []
for dic in info:
    if 'USDT' in dic['symbol']:
        coins.append(dic['symbol'])

coins = ["BTCUSDT", "ETHUSDT", "BTHUSDT", "ETCUSDT", "EOSUSDT", "XRPUSDT", "XLMUSDT", "DOTUSDT", "LTCUSDT", "QTUMUSDT"]

samples = dict()
ref = get_sample("ETHUSDT", start_date="1 Jun, 2020")
for coin in coins:
    sample = get_sample(coin, start_date="1 Jun, 2020")
    if len(sample) != len(ref):
        continue
    
    sample['volume_per_trade'] = sample['volume'] / sample['number of trades']
    sample['raise'] = (sample['close'] - sample['close'].rolling(20).mean() + 2.*sample['close'].rolling(20).std()) / (4.*sample['close'].rolling(20).std())
    sample['vraise'] = (sample['volume_per_trade'] - sample['volume_per_trade'].rolling(20).mean() + 2.*sample['volume_per_trade'].rolling(20).std()) / (4.*sample['volume_per_trade'].rolling(20).std())
    sample.dropna(inplace=True)
    
    samples[coin] = sample

In [4]:
init_balance = 100000

for idx in samples['ETHUSDT'].index:
    # pass the first index
    if idx == samples['ETHUSDT'].index[0]:
        balance = init_balance
        continue
    
    if idx == samples['ETHUSDT'].index[-1]:
        continue
    
    # get make dictionary for trading
    table = []
    for coin in coins:
        this_table = dict()
        this_table['coin'] = coin
        this_table['raise'] = samples[coin].loc[idx, 'raise']
        this_table['vraise'] = samples[coin].loc[idx, 'vraise']
        this_table['rtn'] = samples[coin].shift(-1).loc[idx, 'close'] / samples[coin].loc[idx, 'close']
        table.append(this_table)
    
    # sort w.r.t vraise
    table.sort(key=lambda x: x.get('vraise'), reverse=True)
    table = table[:10]
    
    # sort w.r.t raise
    table.sort(key=lambda x: x.get('raise'))
    table = table[:5]
    
    splited_balance = balance / 5.
    temp_balance = 0.
    for temp in table:
        #temp_balance += splited_balance * temp['rtn']
        if temp['raise'] > 0.25:
            temp_balance += splited_balance * temp['rtn']
        else:
            temp_balance += splited_balance
    balance = temp_balance
    print(f"{idx}: {int(temp_balance)}")
print(f"final balance: {balance}")   

2020-06-11 00:00:00: 93886
2020-06-11 12:00:00: 93886
2020-06-12 00:00:00: 93886
2020-06-12 12:00:00: 93886
2020-06-13 00:00:00: 93886
2020-06-13 12:00:00: 93141
2020-06-14 00:00:00: 93141
2020-06-14 12:00:00: 93141
2020-06-15 00:00:00: 93141
2020-06-15 12:00:00: 93141
2020-06-16 00:00:00: 93506
2020-06-16 12:00:00: 92855
2020-06-17 00:00:00: 92918
2020-06-17 12:00:00: 92624
2020-06-18 00:00:00: 91905
2020-06-18 12:00:00: 91515
2020-06-19 00:00:00: 91003
2020-06-19 12:00:00: 91003
2020-06-20 00:00:00: 90951
2020-06-20 12:00:00: 91396
2020-06-21 00:00:00: 90613
2020-06-21 12:00:00: 90613
2020-06-22 00:00:00: 93122
2020-06-22 12:00:00: 93214
2020-06-23 00:00:00: 93126
2020-06-23 12:00:00: 91180
2020-06-24 00:00:00: 89847
2020-06-24 12:00:00: 89782
2020-06-25 00:00:00: 89051
2020-06-25 12:00:00: 88430
2020-06-26 00:00:00: 87955
2020-06-26 12:00:00: 87936
2020-06-27 00:00:00: 84623
2020-06-27 12:00:00: 84623
2020-06-28 00:00:00: 84623
2020-06-28 12:00:00: 84623
2020-06-29 00:00:00: 84623
2

In [6]:
import pybithumb

df = pybithumb.get_ohlcv("BTC")[-200:]
def get_sample(coin, len=300):
    sample = pybithumb.get_ohlcv(coin)[:-1*len]
    return sample

In [7]:
import pybithumb

coins = ["BTC", "ETH", "ETC", "EOS", "XRP", "XLM", "DOT", "LTC", "QTUM"]

samples = dict()
for coin in coins:
    try:
        sample = pybithumb.get_candlestick(coin, chart_intervals="12h")[-2000:]
    except:
        print(coin)
    sample['raise'] = (sample['close'] - sample['close'].rolling(20).mean() + 2.*sample['close'].rolling(20).std()) / (4.*sample['close'].rolling(20).std())
    sample['vraise'] = (sample['volume'] - sample['volume'].rolling(20).mean() + 2.*sample['volume'].rolling(20).std()) / (4.*sample['volume'].rolling(20).std())
    sample.dropna(inplace=True)
    
    samples[coin] = sample

In [8]:
init_balance = 100000

for idx in samples['ETH'].index:
    # pass the first index
    if idx == samples['ETH'].index[0]:
        balance = init_balance
        continue
    
    if idx == samples['ETH'].index[-1]:
        continue
    
    # get make dictionary for trading
    table = []
    try:
        for coin in coins:
            this_table = dict()
            this_table['coin'] = coin
            this_table['raise'] = samples[coin].loc[idx, 'raise']
            this_table['vraise'] = samples[coin].loc[idx, 'vraise']
            this_table['rtn'] = samples[coin].shift(-1).loc[idx, 'close'] / samples[coin].loc[idx, 'close']
            table.append(this_table)
    
        table.sort(key=lambda x: x.get('vraise'), reverse=True)
        table = table[:10]
    
        # sort w.r.t raise
        table.sort(key=lambda x: x.get('raise'))
        table = table[:5]
    
        splited_balance = balance / 5.
        temp_balance = 0.
        for temp in table:
        	#temp_balance += splited_balance * temp['rtn']
            if temp['raise'] > 0.25:
            	temp_balance += splited_balance * temp['rtn']
            else:
                temp_balance += splited_balance
        balance = temp_balance
        print(f"{idx}: {int(temp_balance)}")
    except:
        pass
print(f"final balance: {balance}")   

2020-10-17 00:00:00: 99697
2020-10-17 12:00:00: 99858
2020-10-18 00:00:00: 99742
2020-10-18 12:00:00: 99219
2020-10-19 00:00:00: 99219
2020-10-19 12:00:00: 98907
2020-10-20 00:00:00: 98399
2020-10-20 12:00:00: 98399
2020-10-21 00:00:00: 98399
2020-10-21 12:00:00: 99693
2020-10-22 00:00:00: 100256
2020-10-22 12:00:00: 99724
2020-10-23 00:00:00: 99079
2020-10-23 12:00:00: 99360
2020-10-24 00:00:00: 99683
2020-10-24 12:00:00: 100139
2020-10-25 00:00:00: 98634
2020-10-25 12:00:00: 98842
2020-10-26 00:00:00: 99581
2020-10-26 12:00:00: 98698
2020-10-27 00:00:00: 98653
2020-10-27 12:00:00: 98349
2020-10-28 00:00:00: 96726
2020-10-28 12:00:00: 96992
2020-10-29 00:00:00: 96887
2020-10-29 12:00:00: 96676
2020-10-30 00:00:00: 96676
2020-10-30 12:00:00: 96676
2020-10-31 00:00:00: 96676
2020-10-31 12:00:00: 96299
2020-11-01 00:00:00: 96242
2020-11-01 12:00:00: 96242
2020-11-02 00:00:00: 94838
2020-11-02 12:00:00: 94838
2020-11-03 00:00:00: 94838
2020-11-03 12:00:00: 94838
2020-11-04 00:00:00: 94838